# Real-Time Facial Deepfake Artifact Detector & Blinking Analyzer
Facial landmarks (MediaPipe) + frequency artifacts (FFT) + blink biometrics -> scikit-learn -> Gradio.

**Run order:** 1 install -> 2 upload project zip -> 3 get dataset -> 4 extract -> 5 train -> 6 launch app. Take a screenshot after each step for your report.

## 1. Install (then Runtime > Restart session if pip warns about numpy)

In [ ]:
!pip -q install mediapipe==0.10.14 'numpy<2' opencv-python-headless scikit-learn pandas matplotlib joblib tqdm 'gradio>=4.44.0' kaggle

## 2. Upload `deepfake-detector.zip` (the project folder zipped)

In [ ]:
from google.colab import files
files.upload()
!unzip -oq deepfake-detector.zip
%cd deepfake-detector
!ls

## 3. Dataset
**Option A - quick (image, frequency features only):** Kaggle *140k Real and Fake Faces* (`xhlulu/140k-real-and-fake-faces`). Upload your `kaggle.json` first.

**Option B - full (video, frequency + blink):** Celeb-DF v2 or FaceForensics++ (see README for access). Put videos in Google Drive and mount it.

In [ ]:
# ---- Option A ----
from google.colab import files
files.upload()  # choose kaggle.json
!mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d xhlulu/140k-real-and-fake-faces -p data --unzip -q
!find data -maxdepth 4 -type d | head -20

In [ ]:
# ---- Option B (videos from Drive) ----
# from google.colab import drive; drive.mount('/content/drive')
# REAL = '/content/drive/MyDrive/celebdf/Celeb-real'; FAKE = '/content/drive/MyDrive/celebdf/Celeb-synthesis'

## 4. Extract features

In [ ]:
# Option A (adjust paths to what `find` printed above)
!python extract_features.py --mode image --real_dir data/real_vs_fake/real-vs-fake/train/real --fake_dir data/real_vs_fake/real-vs-fake/train/fake --limit 2000 --out features_image.csv

In [ ]:
# Option B
# !python extract_features.py --mode video --real_dir $REAL --fake_dir $FAKE --limit 150 --out features_video.csv

## 5. Train + evaluate (saves model + plots in `models/` and `outputs/`)

In [ ]:
!python train.py --mode image --features features_image.csv
# !python train.py --mode video --features features_video.csv

In [ ]:
from IPython.display import Image, display
import glob
for p in sorted(glob.glob('outputs/*.png')): display(Image(p))

## 6. Launch the Gradio app (public link via share=True)

In [ ]:
from app import demo
demo.launch(share=True, debug=True)